# Project – Misinformation Classifier (LIAR)
Goal: Fine-tune DistilBERT to classify claims as True vs False (binary).

## 1. Setup
- Imports
- GPU check
- Seed

In [1]:
import os
import json
import random
import numpy as np

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score

# Paths
PROJECT_ROOT = r"E:\JYU\NLP\NLP\Project"
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

# Seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

torch: 2.5.1+cu121
cuda available: True
gpu: NVIDIA GeForce GTX 1650


## 2. Data
- Load LIAR dataset (HuggingFace)
- Convert 6-way labels into binary (true vs false)
- Create small subsets for quick runs

In [ ]:
from datasets import load_dataset

data_files = {
    "train": r"E:\JYU\NLP\NLP\Project\data\liar\train.tsv",
    "validation": r"E:\JYU\NLP\NLP\Project\data\liar\valid.tsv",
    "test": r"E:\JYU\NLP\NLP\Project\data\liar\test.tsv",
}

column_names = [
    "id", "label", "statement", "subject", "speaker", "speaker_job",
    "state", "party", "barely_true_counts", "false_counts",
    "half_true_counts", "mostly_true_counts", "pants_on_fire_counts",
    "context"
]

ds = load_dataset(
    "csv",
    data_files=data_files,
    delimiter="\t",
    column_names=column_names,
)

print(ds)
print("Train size:", len(ds["train"]))
print("Validation size:", len(ds["validation"]))
print("Test size:", len(ds["test"]))
print("Columns:", ds["train"].column_names)
print("Sample row:", ds["train"][0])

DatasetDict({
    train: Dataset({
        features: ['id', 'label', 'statement', 'subject', 'speaker', 'speaker_job', 'state', 'party', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context'],
        num_rows: 10240
    })
    validation: Dataset({
        features: ['id', 'label', 'statement', 'subject', 'speaker', 'speaker_job', 'state', 'party', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context'],
        num_rows: 1284
    })
    test: Dataset({
        features: ['id', 'label', 'statement', 'subject', 'speaker', 'speaker_job', 'state', 'party', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context'],
        num_rows: 1267
    })
})
Train size: 10240
Validation size: 1284
Test size: 1267
Columns: ['id', 'label', 'statement', 'subject', 'speaker', 'speaker_job', 'state', 'party', 'barely_true_counts', 

In [ ]:
from datasets import ClassLabel

# Define which labels count as "true-ish"
TRUE_LABELS = {"half-true", "mostly-true", "true"}
FALSE_LABELS = {"pants-fire", "false", "barely-true"}

def to_binary(example):
    lab = example["label"]
    if lab in TRUE_LABELS:
        example["binary_label"] = 1
    elif lab in FALSE_LABELS:
        example["binary_label"] = 0
    else:
        example["binary_label"] = -1  # safety
    return example

ds_bin = ds.map(to_binary)

# remove any unexpected labels
ds_bin = ds_bin.filter(lambda x: x["binary_label"] != -1)

print(ds_bin["train"].features)
print("Label distribution (train):")
print(ds_bin["train"].to_pandas()["binary_label"].value_counts())

{'id': Value('string'), 'label': Value('string'), 'statement': Value('string'), 'subject': Value('string'), 'speaker': Value('string'), 'speaker_job': Value('string'), 'state': Value('string'), 'party': Value('string'), 'barely_true_counts': Value('float64'), 'false_counts': Value('float64'), 'half_true_counts': Value('float64'), 'mostly_true_counts': Value('float64'), 'pants_on_fire_counts': Value('float64'), 'context': Value('string'), 'binary_label': Value('int64')}
Label distribution (train):
binary_label
1    5752
0    4488
Name: count, dtype: int64


In [ ]:
# Small subsets for fast debugging
train_small = ds_bin["train"].shuffle(seed=42).select(range(2000))
val_small   = ds_bin["validation"].shuffle(seed=42).select(range(500))
test_small  = ds_bin["test"].shuffle(seed=42).select(range(500))

print("train_small:", len(train_small))
print("val_small:", len(val_small))
print("test_small:", len(test_small))
print("Sample:", train_small[0]["statement"], "=>", train_small[0]["binary_label"])

train_small: 2000
val_small: 500
test_small: 500
Sample: Says Ken Cuccinelli wants to eliminate all forms of birth control. => 0


## 3. Model & Training
- DistilBERT tokenizer + model
- Trainer API
- Evaluate with accuracy + macro F1

In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = "distilroberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Using device:", device)
print("Model:", model_name)

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Using device: cuda
Model: distilroberta-base


In [6]:
from datasets import Dataset

def tokenize_batch(batch):
    # richer input (statement + subject + context)
    sep = tokenizer.sep_token
    texts = [
        f"{s} {sep} subject: {sub} {sep} context: {ctx}"
        for s, sub, ctx in zip(batch["statement"], batch["subject"], batch["context"])
    ]
    return tokenizer(
        texts,
        truncation=True,
        padding="max_length",       # dynamic padding
        max_length=128,     # reduce from 256
    )


train_tok = train_small.map(tokenize_batch, batched=True)
val_tok   = val_small.map(tokenize_batch, batched=True)
test_tok  = test_small.map(tokenize_batch, batched=True)

# Trainer expects the label column to be named "labels"
train_tok = train_tok.rename_column("binary_label", "labels")
val_tok   = val_tok.rename_column("binary_label", "labels")
test_tok  = test_tok.rename_column("binary_label", "labels")

cols_to_keep = ["input_ids", "attention_mask", "labels"]
train_tok.set_format(type="torch", columns=cols_to_keep)
val_tok.set_format(type="torch", columns=cols_to_keep)
test_tok.set_format(type="torch", columns=cols_to_keep)

print(train_tok[0])
print("Tokenized sizes:", len(train_tok), len(val_tok), len(test_tok))

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

{'labels': tensor(0), 'input_ids': tensor([    0,   104,  4113,  3410, 10016,  7309, 29436,  1072,     7,  7677,
           70,  4620,     9,  3113,   797,     4,  1437,     2,  2087,    35,
        17969,  1437,     2,  5377,    35,    10,   637, 34050,  2407,     4,
            2,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1, 

In [7]:
# FULL dataset tokenization (train/val/test)

train_full = ds_bin["train"].map(tokenize_batch, batched=True)
val_full   = ds_bin["validation"].map(tokenize_batch, batched=True)
test_full  = ds_bin["test"].map(tokenize_batch, batched=True)

# Trainer expects label column name = "labels"
train_full = train_full.rename_column("binary_label", "labels")
val_full   = val_full.rename_column("binary_label", "labels")
test_full  = test_full.rename_column("binary_label", "labels")

cols_to_keep = ["input_ids", "attention_mask", "labels"]
train_full.set_format(type="torch", columns=cols_to_keep)
val_full.set_format(type="torch", columns=cols_to_keep)
test_full.set_format(type="torch", columns=cols_to_keep)

print("FULL sizes:", len(train_full), len(val_full), len(test_full))

FULL sizes: 10240 1284 1267


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score
from transformers import DataCollatorWithPadding
from transformers import EarlyStoppingCallback

# ----- class weights (from training labels) -----
y = np.array([int(x) for x in train_full["labels"]])
counts = np.bincount(y, minlength=2).astype(np.float32)
counts = np.maximum(counts, 1.0)  # avoid divide-by-zero if a class is missing
w = (len(y) / (2.0 * counts)).astype(np.float32)
class_weights = torch.tensor(w, dtype=torch.float)

print("Label counts [0,1]:", counts.tolist())
print("Class weights [0,1]:", w.tolist())

# metrics (use macro F1 for grade-relevant reporting)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

# weighted-loss Trainer
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir=rf"E:\JYU\NLP\NLP\Project\models\liar_bin_{model_name.replace('-', '_')}_full",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,

    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    per_device_eval_batch_size=32,

    num_train_epochs=5,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=200,

    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_full,
    eval_dataset=val_full,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

val_metrics = trainer.evaluate(val_full)
print("val metrics:", val_metrics)

Label counts [0,1]: [4488.0, 5752.0]
Class weights [0,1]: [1.1408199071884155, 0.8901251554489136]


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.651295,0.649563,0.644860,0.644790
2,0.600393,0.667567,0.665109,0.655356
3,0.600190,0.879168,0.663551,0.661890
4,0.704609,1.494267,0.642523,0.629470
5,0.457512,1.736380,0.645639,0.639541


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

val metrics: {'eval_loss': 0.8835694789886475, 'eval_accuracy': 0.6643302180685359, 'eval_macro_f1': 0.6625599768781475, 'eval_runtime': 36.8765, 'eval_samples_per_second': 34.819, 'eval_steps_per_second': 1.112, 'epoch': 5.0}


In [9]:
test_metrics = trainer.evaluate(test_full)
print(test_metrics)

{'eval_loss': 0.9625345468521118, 'eval_accuracy': 0.6345698500394633, 'eval_macro_f1': 0.626162941493982, 'eval_runtime': 36.4387, 'eval_samples_per_second': 34.771, 'eval_steps_per_second': 1.098, 'epoch': 5.0}


In [10]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

pred = trainer.predict(test_full)
y_true = pred.label_ids
y_pred = np.argmax(pred.predictions, axis=1)

cm = confusion_matrix(y_true, y_pred)
print("Confusion matrix:\n", cm)
print("\nClassification report:\n", classification_report(y_true, y_pred, digits=4))

Confusion matrix:
 [[307 246]
 [217 497]]

Classification report:
               precision    recall  f1-score   support

           0     0.5859    0.5552    0.5701       553
           1     0.6689    0.6961    0.6822       714

    accuracy                         0.6346      1267
   macro avg     0.6274    0.6256    0.6262      1267
weighted avg     0.6327    0.6346    0.6333      1267



In [12]:
# Save confusion matrix + classification report
import os
import json
from sklearn.metrics import classification_report, confusion_matrix

out_dir = r"E:\JYU\NLP\NLP\Project\results"
os.makedirs(out_dir, exist_ok=True)

cm = confusion_matrix(y_true, y_pred)
report_dict = classification_report(y_true, y_pred, digits=4, output_dict=True)

with open(os.path.join(out_dir, "liar_confusion_matrix.json"), "w", encoding="utf-8") as f:
    json.dump({"confusion_matrix": cm.tolist()}, f, indent=2)

with open(os.path.join(out_dir, "liar_classification_report.json"), "w", encoding="utf-8") as f:
    json.dump(report_dict, f, indent=2)

print("Saved:",
      os.path.join(out_dir, "liar_confusion_matrix.json"),
      "and",
      os.path.join(out_dir, "liar_classification_report.json"))

Saved: E:\JYU\NLP\NLP\Project\results\liar_confusion_matrix.json and E:\JYU\NLP\NLP\Project\results\liar_classification_report.json


## 4. Save Outputs
- Save metrics to: E:\JYU\NLP\NLP\Project\results\
- Save label mapping / rules used for binary conversion

In [11]:
import json, os
from datetime import datetime

out_dir = r"E:\JYU\NLP\NLP\Project\results"
os.makedirs(out_dir, exist_ok=True)

run = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "model": model_name,
    "task": "LIAR binary (true-ish vs false-ish)",
    "label_rules": {
        "1_trueish": sorted(list(TRUE_LABELS)),
        "0_falseish": sorted(list(FALSE_LABELS)),
    },
    "train_size": len(train_full),
    "val_size": len(val_full),
    "test_size": len(test_full),
    "val_metrics": val_metrics,
    "test_metrics": test_metrics,
}

path = os.path.join(out_dir, "liar_binary_metrics.json")
with open(path, "w", encoding="utf-8") as f:
    json.dump(run, f, indent=2)

print("Saved:", path)

Saved: E:\JYU\NLP\NLP\Project\results\liar_binary_metrics.json
